# PICasso Benchmark Monitoring Dashboard

Live progress tracking for two-phase benchmarking (Raw LLM vs Framework).

This notebook provides real-time monitoring of:
- Phase 1 (Raw LLM baseline) results
- Phase 2 (PICasso Framework) results  
- Pilot validator catches
- Auto-correction attempts
- Stage-wise validation metrics

**Usage:**
1. Run all cells to set up monitoring
2. Start the benchmark in a separate terminal: `python run_9_problem_test.py`
3. The dashboard will update automatically every 10 seconds


In [5]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import time
from IPython.display import display, clear_output, HTML
import numpy as np

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# File paths
RAW_CSV = Path("hf_inference_workflow/output/results/raw_llm_results.csv")
FRAMEWORK_CSV = Path("hf_inference_workflow/output/results/framework_results.csv")

print("✅ Setup complete!")
print(f"Monitoring:")
print(f"  Raw LLM: {RAW_CSV}")
print(f"  Framework: {FRAMEWORK_CSV}")


✅ Setup complete!
Monitoring:
  Raw LLM: hf_inference_workflow/output/results/raw_llm_results.csv
  Framework: hf_inference_workflow/output/results/framework_results.csv


In [7]:
def monitor_progress(refresh_interval=10, max_iterations=360):
    """
    Monitor benchmark progress with live updates.
    
    Args:
        refresh_interval: Seconds between updates (default: 10)
        max_iterations: Maximum refresh cycles (default: 360 = 1 hour at 10s interval)
    """
    iteration = 0
    
    try:
        while iteration < max_iterations:
            clear_output(wait=True)
            
            print("="*70)
            print("PICASSO BENCHMARK MONITORING DASHBOARD")
            print("="*70)
            print(f"Refresh: {iteration + 1}/{max_iterations} (every {refresh_interval}s)")
            print("="*70)
            print()
            
            # Phase 1: Raw LLM Results
            if RAW_CSV.exists():
                raw_df = pd.read_csv(RAW_CSV)
                print("📊 PHASE 1: RAW LLM (Baseline)")
                print(f"  Total Samples: {len(raw_df)}")
                if 'passed' in raw_df.columns:
                    success_count = raw_df['passed'].sum()
                    print(f"  Success: {success_count} / {len(raw_df)} ({success_count/len(raw_df)*100:.1f}%)")
                print()
            else:
                print("📊 PHASE 1: RAW LLM - No data yet")
                print()
            
            # Phase 2: Framework Results
            if FRAMEWORK_CSV.exists():
                fw_df = pd.read_csv(FRAMEWORK_CSV)
                print("🚀 PHASE 2: PICASSO FRAMEWORK")
                print(f"  Total Samples: {len(fw_df)}")
                
                if 'passed' in fw_df.columns:
                    success_count = fw_df['passed'].sum()
                    print(f"  Success: {success_count} / {len(fw_df)} ({success_count/len(fw_df)*100:.1f}%)")
                
                # Pilot catches
                if 'pilot_error' in fw_df.columns:
                    pilot_catches = fw_df['pilot_error'].sum()
                    print(f"  Pilot Catches: {pilot_catches}")
                
                # Auto-corrections
                if 'auto_corrected' in fw_df.columns:
                    auto_corrects = fw_df['auto_corrected'].sum()
                    print(f"  Auto-Corrected: {auto_corrects}")
                
                print()
                print("  Validation Stages:")
                if 'pnr_passed' in fw_df.columns:
                    print(f"    PNR Pass: {fw_df['pnr_passed'].sum()} ({fw_df['pnr_passed'].mean()*100:.1f}%)")
                if 'drc_passed' in fw_df.columns:
                    print(f"    DRC Pass: {fw_df['drc_passed'].sum()} ({fw_df['drc_passed'].mean()*100:.1f}%)")
                if 'sax_passed' in fw_df.columns:
                    print(f"    SAX Pass: {fw_df['sax_passed'].sum()} ({fw_df['sax_passed'].mean()*100:.1f}%)")
                if 'functional_passed' in fw_df.columns:
                    print(f"    Functional Pass: {fw_df['functional_passed'].sum()} ({fw_df['functional_passed'].mean()*100:.1f}%)")
                if 'loss_target_met' in fw_df.columns:
                    print(f"    Loss Target Met: {fw_df['loss_target_met'].sum()} ({fw_df['loss_target_met'].mean()*100:.1f}%)")
                
                print()
            else:
                print("🚀 PHASE 2: PICASSO FRAMEWORK - No data yet")
                print()
            
            print("="*70)
            print("Press Interrupt (Kernel → Interrupt) to stop monitoring")
            print("="*70)
            
            time.sleep(refresh_interval)
            iteration += 1
            
    except KeyboardInterrupt:
        print("\n\n✅ Monitoring stopped by user")

# Start monitoring (comment out to prevent auto-start)
monitor_progress(refresh_interval=10)


PICASSO BENCHMARK MONITORING DASHBOARD
Refresh: 360/360 (every 10s)

📊 PHASE 1: RAW LLM - No data yet

🚀 PHASE 2: PICASSO FRAMEWORK - No data yet

Press Interrupt (Kernel → Interrupt) to stop monitoring


In [8]:
def plot_comparison():
    """Compare Phase 1 (Raw LLM) vs Phase 2 (Framework) results."""
    
    if not RAW_CSV.exists() or not FRAMEWORK_CSV.exists():
        print("❌ Results files not found. Run the benchmark first.")
        return
    
    raw_df = pd.read_csv(RAW_CSV)
    fw_df = pd.read_csv(FRAMEWORK_CSV)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Overall success rate
    ax = axes[0, 0]
    if 'passed' in raw_df.columns and 'passed' in fw_df.columns:
        success_rates = [
            raw_df['passed'].mean(),
            fw_df['passed'].mean()
        ]
        bars = ax.bar(['Phase 1\n(Raw LLM)', 'Phase 2\n(Framework)'], success_rates,
                      color=['#ff6b6b', '#4ecdc4'])
        ax.set_ylabel('Success Rate', fontsize=12, fontweight='bold')
        ax.set_title('Overall Pass Rate Comparison', fontsize=14, fontweight='bold')
        ax.set_ylim([0, 1.0])
        
        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height*100:.1f}%',
                   ha='center', va='bottom', fontweight='bold')
    
    # Plot 2: Stage-wise pass rates (Framework only)
    ax = axes[0, 1]
    stages = []
    fw_rates = []
    
    stage_cols = {
        'PNR': 'pnr_passed',
        'DRC': 'drc_passed', 
        'SAX': 'sax_passed',
        'Functional': 'functional_passed',
        'Loss Target': 'loss_target_met'
    }
    
    for stage_name, col_name in stage_cols.items():
        if col_name in fw_df.columns:
            stages.append(stage_name)
            fw_rates.append(fw_df[col_name].mean())
    
    if stages:
        bars = ax.bar(stages, fw_rates, color='#95e1d3')
        ax.set_ylabel('Pass Rate', fontsize=12, fontweight='bold')
        ax.set_title('Framework Stage-wise Performance', fontsize=14, fontweight='bold')
        ax.set_ylim([0, 1.0])
        ax.tick_params(axis='x', rotation=15)
        
        # Add value labels
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height*100:.1f}%',
                   ha='center', va='bottom', fontsize=9)
    
    # Plot 3: Retry attempts distribution
    ax = axes[1, 0]
    if 'retry_attempts' in fw_df.columns:
        retry_data = fw_df['retry_attempts'].value_counts().sort_index()
        ax.bar(retry_data.index, retry_data.values, color='#f38181', alpha=0.7)
        ax.set_xlabel('Retry Attempts', fontsize=12, fontweight='bold')
        ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
        ax.set_title('Retry Distribution (Framework)', fontsize=14, fontweight='bold')
        ax.set_xticks(range(int(fw_df['retry_attempts'].max()) + 1))
    
    # Plot 4: Error handling mechanisms
    ax = axes[1, 1]
    mechanisms = []
    counts = []
    
    if 'pilot_error' in fw_df.columns:
        pilot_catches = fw_df['pilot_error'].sum()
        mechanisms.append('Pilot\nCatches')
        counts.append(pilot_catches)
    
    if 'auto_corrected' in fw_df.columns:
        auto_corrects = fw_df['auto_corrected'].sum()
        mechanisms.append('Auto-\nCorrects')
        counts.append(auto_corrects)
    
    if 'retry_attempts' in fw_df.columns:
        llm_fixes = fw_df['retry_attempts'].gt(0).sum()
        if 'auto_corrected' in fw_df.columns:
            llm_fixes -= auto_corrects  # Don't double count
        mechanisms.append('LLM\nFixes')
        counts.append(llm_fixes)
    
    if mechanisms:
        bars = ax.bar(mechanisms, counts, color=['#ffd93d', '#6bcf7f', '#a8e6cf'])
        ax.set_ylabel('Count', fontsize=12, fontweight='bold')
        ax.set_title('Error Handling Mechanisms', fontsize=14, fontweight='bold')
        
        # Add value labels
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{int(height)}',
                   ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\n" + "="*70)
    print("SUMMARY STATISTICS")
    print("="*70)
    print(f"Phase 1 (Raw LLM):")
    print(f"  Samples: {len(raw_df)}")
    if 'passed' in raw_df.columns:
        print(f"  Success Rate: {raw_df['passed'].mean()*100:.1f}%")
    
    print(f"\nPhase 2 (Framework):")
    print(f"  Samples: {len(fw_df)}")
    if 'passed' in fw_df.columns:
        print(f"  Success Rate: {fw_df['passed'].mean()*100:.1f}%")
        if 'passed' in raw_df.columns:
            improvement = (fw_df['passed'].mean() - raw_df['passed'].mean()) * 100
            print(f"  Improvement: +{improvement:.1f} percentage points")
    
    if 'pilot_error' in fw_df.columns:
        print(f"  Pilot Errors Caught: {fw_df['pilot_error'].sum()}")
    if 'auto_corrected' in fw_df.columns:
        print(f"  Auto-Corrections: {fw_df['auto_corrected'].sum()}")
    print("="*70)


## Usage Instructions

**To monitor a running benchmark:**
```python
monitor_progress(refresh_interval=10)
```

**To visualize completed results:**
```python
plot_comparison()
```

**Tips:**
- Run the monitoring in a separate notebook tab while benchmark runs
- Interrupt monitoring with Kernel → Interrupt
- Refresh interval can be adjusted (5-30 seconds recommended)
- Results auto-save every 10 samples, so monitoring updates incrementally
